# UK Postcode Lookup — postcodes.io

Demonstrates `lookup_postcode()` and `lookup_outcode()` from the `airgap_geo`
library, which query the self-hosted **postcodes.io** API for UK postcode and
outcode (district) metadata.

## Prerequisites

| Service          | Default port | Role                       |
| ---------------- | ------------ | -------------------------- |
| postcodes.io API | 8000         | UK postcode / outcode data |


______________________________________________________________________

## Start Services (optional)

Run the cell below to start the combined stack and wait until postcodes.io
responds. **Skip if the services are already running.**


In [ ]:
import pathlib
import subprocess
import time

import requests as _requests

from airgap_geo.settings import POSTCODES_URL


def _find_repo_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


_REPO_ROOT = _find_repo_root(pathlib.Path().resolve())
_COMPOSE_FILE = _REPO_ROOT / "docker" / "docker-compose.yml"
_ENV_FILE = _REPO_ROOT / ".env"

_HEALTH_TIMEOUT = 120

_ENDPOINTS = {
    "postcodes.io": POSTCODES_URL,
}


def start_services(timeout: int = _HEALTH_TIMEOUT) -> None:  # noqa: D103
    print(f"Starting stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "up",
            "-d",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    ready = {name: False for name in _ENDPOINTS}
    deadline = time.monotonic() + timeout
    print(f"\nPolling services (timeout {timeout}s) ...")

    while time.monotonic() < deadline:
        for name, url in _ENDPOINTS.items():
            if ready[name]:
                continue
            try:
                _requests.get(url, timeout=3)
                ready[name] = True
                print(f"  \u2713  {name} is up  ({url})")
            except Exception:
                pass
        if all(ready.values()):
            break
        time.sleep(3)

    still_down = [n for n, ok in ready.items() if not ok]
    if still_down:
        print(f"\n[WARNING] Timed out waiting for: {', '.join(still_down)}")
    else:
        print("\nAll services are up and ready.")


start_services()

## Setup — Imports and Configuration


In [ ]:
import importlib
import pprint

import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import lookup_outcode, lookup_postcode  # noqa: E402
from airgap_geo.settings import POSTCODES_URL  # noqa: E402

client = httpx.AsyncClient()

print(f"postcodes.io : {POSTCODES_URL}")

## Health Check


In [ ]:
try:
    r = requests.get(POSTCODES_URL, timeout=5)
    print(f"postcodes.io  \u2713  HTTP {r.status_code}  ({POSTCODES_URL})")
except Exception:
    print(f"postcodes.io  \u2717  unreachable  ({POSTCODES_URL})")

______________________________________________________________________

## 1 — Full Postcode Lookup

`lookup_postcode(postcode)` returns the full metadata for a UK postcode, including
coordinates, administrative geography, parliamentary constituency, and more.


In [ ]:
pc_result = await lookup_postcode("SW1A 2AA", client)
pprint.pprint(pc_result)

______________________________________________________________________

## 2 — Outcode (District) Lookup

`lookup_outcode(outcode)` returns aggregated metadata for an outward code (the
first part of a UK postcode, e.g. `"SW1A"`).


In [ ]:
oc_result = await lookup_outcode("SW1A", client)
pprint.pprint(oc_result)

______________________________________________________________________

## 3 — Comparison Table


In [ ]:
_PC_FIELDS = [
    "postcode",
    "latitude",
    "longitude",
    "admin_district",
    "admin_ward",
    "parliamentary_constituency",
    "ccg",
    "nuts",
]

rows = []
for field in _PC_FIELDS:
    rows.append(
        {
            "Field": field,
            "SW1A 2AA": pc_result.get(field, "-"),
            "SW1A (outcode)": oc_result.get(field, "-"),
        }
    )

pd.DataFrame(rows).set_index("Field")

______________________________________________________________________

## 4 — Error Handling

Invalid postcodes and outcodes return an empty dict rather than raising exceptions.


In [ ]:
bad_pc = await lookup_postcode("ZZ9 9ZZ", client)
print(f"Invalid postcode  \u2192 {bad_pc!r}  (expected: {{}})")

bad_oc = await lookup_outcode("ZZ9", client)
print(f"Invalid outcode   \u2192 {bad_oc!r}  (expected: {{}})")

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:  # noqa: D103
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()